# 第 10 章实验：合成迁移学习与 Grad-CAM
先在条纹图上学习通用边缘特征，再冻结特征层训练新分类头。全程随机初始化且不下载权重。

第一格会打印当前用的设备（有 NVIDIA 显卡并且装了带 CUDA 的 PyTorch 时是 `cuda`，否则是 `cpu`）。数据直接放在内存里的张量中，没有用 `DataLoader`。Grad-CAM 那一格和最后一格会调用你在 `作业_冻结策略与热图归一化.py` 里写的函数，作业还没完成时只打印提示。

In [ ]:
import importlib.util
import os
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# 图里的中文：Windows 自带的字体放在前面，找不到再试 macOS 和 Linux 上的
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "PingFang SC", "Heiti SC", "Noto Sans CJK SC", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

# 设备：有 NVIDIA 显卡就用 CUDA，没有就用 CPU。本实验很小，在 CPU 上也只要几秒到几十秒
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| 设备:", device, "|",
      torch.cuda.get_device_name(0) if device.type == "cuda" else "没有检测到 CUDA，用 CPU 运行")

CHAPTER_NAME = "10_迁移学习"
HOMEWORK_FILE = "作业_冻结策略与热图归一化.py"


def load_homework():
    """载入本章的作业文件。设置了环境变量 COURSE_IMPL_DIR 时改从那个目录载入（维护课程时用）。"""
    here = Path.cwd().resolve()
    chapter = None
    for p in [here, *here.parents]:
        if p.name == CHAPTER_NAME:
            chapter = p
            break
        if (p / CHAPTER_NAME).is_dir():
            chapter = p / CHAPTER_NAME
            break
    if chapter is None:
        raise FileNotFoundError("请在课程目录内启动 Jupyter")
    base = chapter
    override = os.environ.get("COURSE_IMPL_DIR")
    if override:
        root = Path(override)
        if not root.is_absolute() and not (root / CHAPTER_NAME).is_dir():
            root = chapter.parent / root   # 相对路径按课程根目录算
        base = root / CHAPTER_NAME
    spec = importlib.util.spec_from_file_location("homework", base / HOMEWORK_FILE)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def attempt(fn, *args, **kwargs):
    """调用作业里的函数；还没实现时打印提示并返回 None，不中断整个 Notebook。"""
    try:
        return fn(*args, **kwargs)
    except NotImplementedError as exc:
        print(f"先完成作业里的 {getattr(fn, '__name__', type(fn).__name__)}：{exc}")
        return None


hw = load_homework()
print("作业文件：", HOMEWORK_FILE)

In [ ]:
# 参数：改这里，然后从头运行
torch.manual_seed(10)
pretrain_steps = 80
transfer_steps = 50
image_size = 24

In [ ]:
def make_stripes(n, inverted=False):
    # 在 CPU 上生成，调用的地方再搬到 device：同一个种子在任何机器上都得到同样的图
    images = torch.zeros(n, 1, image_size, image_size)
    labels = torch.arange(n) % 2
    for i, label in enumerate(labels):
        if int(label) == 0:
            images[i, 0, :, 7:11] = 1.0
        else:
            images[i, 0, 7:11, :] = 1.0
    images += 0.12 * torch.randn_like(images)
    images = images.clamp(0, 1)
    return (1 - images if inverted else images), labels

class TinyTransferNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(nn.Conv2d(1, 6, 3, padding=1), nn.ReLU(), nn.Conv2d(6, 8, 3, padding=1), nn.ReLU())
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Linear(8, 2)
    def forward(self, x):
        feature_map = self.features(x)
        return self.head(self.pool(feature_map).flatten(1))

model = TinyTransferNet().to(device)
source_x, source_y = make_stripes(64)
source_x, source_y = source_x.to(device), source_y.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.03)
for _ in range(pretrain_steps):
    optimizer.zero_grad(); loss = F.cross_entropy(model(source_x), source_y); loss.backward(); optimizer.step()
print('source accuracy:', float((model(source_x).argmax(1) == source_y).float().mean()))

In [ ]:
for parameter in model.features.parameters():
    parameter.requires_grad = False
model.head = nn.Linear(8, 2).to(device)   # 新建的层在 CPU 上，要搬到和模型同一个设备
target_x, target_y = make_stripes(12, inverted=True)
target_x, target_y = target_x.to(device), target_y.to(device)
optimizer = torch.optim.Adam(model.head.parameters(), lr=0.05)
for _ in range(transfer_steps):
    optimizer.zero_grad(); loss = F.cross_entropy(model(target_x), target_y); loss.backward(); optimizer.step()
trainable = [name for name, p in model.named_parameters() if p.requires_grad]
accuracy = float((model(target_x).argmax(1) == target_y).float().mean())
print('trainable parameters:', trainable)
print('target small-set accuracy:', accuracy)

## Grad-CAM
对最后卷积层的激活保留梯度。热图是相关性线索，不是像素级真值或因果证明。

这一格会把归一化之前的最大值打印出来。留意它的大小：单张归一化之后，热图的最大值总是 1，原始响应有多弱就看不出来了。教程 10.5 节和 10.11 节的第 3 个破坏实验用的就是这个现象。

In [ ]:
saved = {}
def capture_activation(module, inputs, output):
    saved['activation'] = output
    output.retain_grad()
handle = model.features[2].register_forward_hook(capture_activation)
sample = target_x[:1].clone().requires_grad_(True)
model.zero_grad()
logits = model(sample)
class_id = int(logits.argmax(1))
logits[0, class_id].backward()
activation = saved['activation']
weights = activation.grad.mean(dim=(2, 3), keepdim=True)
raw_cam = (weights * activation).sum(dim=1, keepdim=True).detach()   # 还没有做 ReLU 和归一化，shape (1, 1, H, W)
print('归一化之前热图的最大值:', float(raw_cam.max()))
cam = attempt(hw.normalize_heatmap, raw_cam)
if cam is None:
    # 作业还没完成：先只截掉负值。下面画图时 matplotlib 会按这一张图的最小最大值上色，看到的图案是一样的
    cam = raw_cam.relu()
cam = F.interpolate(cam, size=sample.shape[-2:], mode='bilinear', align_corners=False)
handle.remove()
print('logits:', tuple(logits.shape), 'activation:', tuple(activation.shape), 'cam:', tuple(cam.shape))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(sample.detach()[0, 0].cpu(), cmap='gray'); axes[0].set_title('目标域样本')
axes[1].imshow(sample.detach()[0, 0].cpu(), cmap='gray'); axes[1].imshow(cam.detach()[0, 0].cpu(), cmap='jet', alpha=0.55); axes[1].set_title(f'Grad-CAM 类别 {class_id}')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

### 破坏实验建议
把训练图的背景与类别绑定，再用相反背景测试；比较准确率和热图是否转向背景。该干预比单看一张热图更能发现 shortcut learning。

## 接上你的作业：冻结策略
上面是手写循环冻结特征层的。这一格用你在 `作业_冻结策略与热图归一化.py` 里写的 `configure_trainable_parameters` 做同一件事，再解冻最后一个卷积层，打印函数返回的名单和模型里实际可训练的参数，两者应该一致。作业还没完成时这一格只打印提示。

In [ ]:
def show_trainable(prefixes):
    names = attempt(hw.configure_trainable_parameters, model, prefixes)
    if names is not None:
        actual = [name for name, p in model.named_parameters() if p.requires_grad]
        print(prefixes, '-> 返回', names, '| 和实际可训练的参数一致:', names == actual)
    return names

if show_trainable(('head',)) is not None:
    show_trainable(('features.2', 'head'))   # 逐块解冻：多解冻最后一个卷积层
    show_trainable(('head',))                # 再冻回去
    print('改了 requires_grad 之后要重建优化器，原因见教程 10.2 节的实测表。')